# 🫀 AI 的資料前處理基礎：動手做做看
**兩小時工作坊｜給零程式背景的醫學生**

> 今天不教你寫程式。程式都已經寫好了，你只需要 **按 ▶ 執行（或 Shift+Enter）**，並在標示 **👉 你的任務** 的地方動手。
> 目標是讓你親眼看到：**資料怎麼處理，會直接決定 AI 學到什麼、你能不能相信它。**

**今天的流程**

| 單元 | 內容 | 你會看到 |
|---|---|---|
| 0 | 準備環境 | 授權 Google Drive |
| 1 | 資料如何才能「餵進去」 | 髒資料讓模型壞掉；你手動清理 vs 程式自動清理 |
| 2 | 資料怎麼分配 | 訓練／驗證／測試、交叉驗證、資料洩漏 |
| 3 | 前處理如何影響訓練 | One-hot、標準化、擴增、資料不平衡 |
| 4 | 推論 | 新病人進來，前處理必須跟訓練時一模一樣 |

**資料來源**：UCI Heart Disease（Cleveland 資料集），Detrano et al., *Am J Cardiol* 1989；ISLR 整理版；CC BY 4.0。
本 notebook 裡的「錯誤」是老師**刻意製造**的，用於教學；模型僅供學習，**不可用於臨床**。

## 單元 0｜準備環境

In [ ]:
#@title 0-1　準備環境（會跳出 Google Drive 授權視窗，請按「允許」）
import os, sys, io, json, time, base64, zlib, warnings, urllib.request
import numpy as np, pandas as pd
warnings.filterwarnings("ignore")

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    WORKDIR = "/content/drive/MyDrive/AI_preprocessing_class"
else:                                            # 不在 Colab 時（老師測試用）
    WORKDIR = os.path.join(os.path.expanduser("~"), "AI_preprocessing_class")
os.makedirs(WORKDIR, exist_ok=True)
print("✅ 你的工作資料夾：", WORKDIR)
print("   之後所有檔案都會存在這裡，打開 Google 雲端硬碟就看得到。")

In [ ]:
#@title 0-2　備援資料（網路出問題時才會用到，不用展開）
_EMBEDDED = {
    "heart_dirty_small.csv":
        "eNqtVstu2zAQvPcr/AELh1yKlHgqjKJAC+SFNpeeDMVmHCG2bFhum/x9hxSl6sGcWhiwBEGzj9nZoU7lpXL1ZV1tqdw5atwrbZ5dc1mfyqqmM+6qerd+POHpcU9Pj014tnabHR3K1/XzmdxrWe/ouN+eXPlCzf54crQp6fJc7mlbNa5s3Id7ITRlkp7codw7qo/15e1UbfCGVIJYZIR/kjnjKpcZSVzr4xkv4+LRloyigF19/3Fz/3B3s3r4+qlF64jO8ojGPar85c5N9QjEm2sQgiWZrA3R55aC9DIzEe7TymVBPEmOHJlokWXzdjhdjgeQtglwafIWLT2aA3aWWqH3mBqdg62q7jpXCgCEya3vOdG5MpRxKjlKYjYRXbTXCRZJtZ3n1UBKDm9L3XJmZlitySSbDozHqiNl4X/eti78yJO8LQ6HL7s4b1Si/9Y/j2JJJ8hjTflSxNFpH0csNSnEGHYBYvRQdT0c7OVL202+lZ0ObYzglrKCUpIVLbIId0ueaaZAYpVKzNCXjHml7tuOKDtATWgHTkWhG5Ect2UynEYbr9RuTXxWg5oVflO6pVAETSSDoHQlVGTcxhJGhEm/p8mRcxw5d7VnHqwgvOmyIopfS4b4RjEW4MtzZ3u8n1l646SUvo5kF2g/8mBC+WP8Yuue3OYSw7Dy8pnNUPjV41iHejwdwhbMROAJ66Q7Mw5WsRH2ZXCwnfkO/Y8QilNNWNgk5y1et7YzWWLhsXCuTpCLIfxn/VIff9eYR7QCo9Ob4PnWRYJDTZxBFLur7XWUlK/i9mo1nWiIgrUwvSxvB3GwQ9oM90IGTc0jAA6buFldfx6bMFjjrodO1KMGCtmbwETToDDvUpvoQEk5FiMP630EC8tFXCirRb9RM7yFGRdJN/FWLgduImMJgwYYUnnPF/xRkGUDFbTHZ48P6Rk2iymmdgkmImx/+raFzE3l3ysA1VolZ6BAYbuKVscTmOl2FXHQL4ImiDP+q0PGsauUlzFO3vTqsR9bZ+Em2FhMe/cN+upSQ2/mvQO0UKFW77YyKFYlZOP3O7PJbyavvK56jofReOgYh06pVsRjxNPdalZONfcjJIfgUh8P8BqGDwe/aM+j0PxHeqpe3TYWnuseO6pZZMOal0UgYTRoK3rWhg17nWditKVq1PAfqZkQew==",
    "heart_dirty_full.csv":
        "eNqtXMuOXrcN3vcp5gGO/4q6axUMigYNkJkEaTZeGRN77AziGRsZt03evqRE6ehCzaLuJg4S80iiyI/kR+r/fPfl4f7py5uHd8fdh/vj+f6P4+2v989f3ny+e3g6fsd/e3j68OaXz/hfP3083v/ynP/bm/u3H47Huz/e/Pr7cf/H3dOH49PHd5/v7347nj9++nx/vL07vvx69/F49/B8f/d8/5cflYLDGxT5eH98+fPzw1v8n2DdoY054NAHOHWoQ1/MYfDP9w9/3L87nj6RnD58KHJ3z38+fv7y6RF3/PYAj389ehI6QEX8CFzwcyh+++l3/OvHn/fPJG424hoFdSri+Cfu4eLx3zUe79/3vz8//IIS/Al7GP7E06cnPOvDE23eoGjetDogBvynwfVp809l/bJ7d1g43t9X6XZ0ksYP5+WDpm9cLG5ikvaH88fjIEj7Nr4sGyL9eYlZ8LYXDIfXddnrf76++fHnH26uf/7ub6hzlPexLOxV3rbHbeu6MJ85Hi7UDyyKM86WDXi6PIUf6HZ+mzeQ2m2Py2et8blt4HPjv69qB3U4I90cnUBVq3H4p7lAVvz6CaBTyJ+ApMsZbFEi7WIwPNCkfFbBbXcDKBwuWXW0AZPPYLJ0f3dg8t2JVuPL5q1m7dGfZeW8b42+gpYlah86/blUTjCvbA9rj8XiyHC8YcMxTbJT2hMJu8PpddtoopDS1eOHv777PouBz+q7uOkjV+/u39+//cI78U3/t93HHIlH3onN+vPrKcKBN7OcAqrb1m/srh4N2O6sx9QP5H9e9Lp4osVPtx2uL7giXb4ySmfjR03bdIgu76vnQpZdj60RJu0Ek3RmAPYZm5Eu9vamSA6tte3459c/fve36++zpnVkV8kXBsNmSc6QnGgs0XYoMS1ZlKxtL3wqCfdyCYyt2dRM1pKArdod+HnhmjI+MrqbLJw9VPqE75wFt/Gq7YMOAeWqv2Erm71UBzrBt+tFE8pZRvegRT/T8fANIJrmSOWabxmyBjO+TqJoXyK2kXSGxWojwB7WS6NurKg0NBR7MbpoDWxWuZa8w6CVpeP90WNa8woHfN9wRoWyrm6XdT2uG1BaMS5kI7tY+b4x2nu7Wgwubwy7lYuztlkUjS2JpkJZRA1I8MvnRwYmOvlP5wbIS/CvWiuDsqnQmDIor8EYg67VO0TRQ0ReZcPmvjXJBM5CVDZzt7myuIllZDKBPQU0Hb8GlW/6oKLwejFaLh+4yv5lHB8/+xpB4hiTK6yX+IQb8ZvsAH1Wa/6WLSHCLIaEGvOwXkM2flPMCDUGWW5UJCZT3skrZ8dzzfazMtHxFkXirqr3NJzM+UBqVwgtG+ujY17fHEHM59DsjWKc4NBo13OfJjyfG0oYARc28RBzZQ/yyQ15D+cjvuYEo/NYLyI1rqqbZNZdNj8hG0NQcsKVASmMHbdcu7T1SAi9QzvD1h8ZK0Xrx3h63vuEHBYCJ4OhpcMwbMDlVPLm+vu/z24PqcpqTgVmqMZTyxk8pvaQYpeJqHVdTbu+ecHtVT6BaoAz2xtu24ppDN1c6nPQYfWiNCcngqaGRRSNMl45t0H6nL5W97bNUQSDwSRXSsAQbjBrKHkQFW6l8ipw89PyjdAM53a88wbXvhqdaLVo8xakMgRqWNdDwFk/kNop+h0gzujAhutqImym20cNovHcHD+3pAqloJq765Q3SAFJFXu7WpCaUMY1dK1ILWwcMcj6zmyvTu3pM+S0YG3GbNKfWeFkuPHq8fEfH/AcNTeDdoEC1mKsR/VJebQr3zGq2D0EJRoiuo+cxpMHVeMF81IdiaaGx7yZgzbeRWQDsCW/jHN+CSQdmidMsAORg42jP82aHfuzBJnRvuZoGXMr2ncr5zICoRzN70pyIdSW0XwDNtdAfIfL6fFvWr/ZBLAhBWYBlhME2KAuHlnXSGN7EqK/u5CZmyvp8lwuYqFzYN0p/462/7rswBCRsaFvfEHQlFpJJGScWGB6t0mWtY09BFxKXd4nTcHt8LfEjlLiBBl/MSuT1UfJQiMQYKe+sCmNsvZ44xaa6Qqmj6mMg03UNCo2669lxohEIYkVOIUAy6YfVbu8cfP4P2QAydkqHx1yNcpH7zZfrj5i4BWDJ0KGbpuPmQBYltcU8HfJkvaVgDId9TPImwPT5RtJbZoL8SgXhVg0eyHRinRs25hOaMVof2ckPtAvfTlutOurUYG0iF4sbnK0HO5rlcTyRGI40awc27jzsmTsFD3Uz/GMEeCabOaYYtrfTo0vpicf1nWT2lg282vlelNHUd3+8BMmgDm6JaDCfUNtKj345WobSXfc6kJJQ7MtOrTPCaUQGFPGtRuxEK2w7BvtYpa8MOXUTgrvRIVb/kTJzAXaIzmiZ8W4jOrTlZbmWnrgjH4gcd+sdM7tIp658pK+sE1ieojYKVfzUJkIPaQ40iciAaSUKRWYqRhZTHjD7hN4OzHCoCJLc6KGSHWBDDbLXaT2hZ4mNgRT0HMaa2UCShHGijESd6v5JuLifIpkM0BKKqTozBgbneQ/gFZuFpoxMqpppmUFMbNvC0T0HTMR09Nh7Vg891Wk9q6go1GyrOs51Z6ma2mgScVe5VyUmAknVEOK9l2ZLy/S4nn9nb3arpTzemvwUDoqNwJOukuqmKMa5khfSI01EXIh1apJ4GRO+AQq2sheFxtVWgrSXWFBQXvHQ6A1GsXxwvrG36yf0K01smY2NnFzxHa+z5YAJGyat42Wy6CfiiLGZg4RQnYXcsgVgI3Pt3Riw32Rc53wPyCvORssKTQVSrfgW1XaWeK/nn57+vSfp9zbqlScdJIgMjm29AigVRWr86KWTEtHp7KeFMjLxr4byVGzyJeenkw+QSPqC3M8OOBrlEYjlfNw486Oli5QLaezgKmXjbtmQW2qOeahJaxHPPVmyx7aart5N1ZGEeqXCDyYUozWKmeVZ2HeK5DaJW7HIANjSL5Exz2DdXnXtHgGmphLWWi4XUvhvhgBLBet23Wia7a1Sxi+Kk5Rj8WLWtekthjZeEhtNnvfeuxdNw+6voX2L9RC1OPfdUV17fOIuSYpaNdMJyqrBg9vZGkQWQDdJSicXdgZLYu8lgwO8z+t2WJqJT0yaLcki8rcNCJ59kCrs4y57Xdt26qrqXpo1w1Tdphl3Ym1V+OhNRNgqoKU4CWIwU42VMcDFzqnltXOJ+lwBLFqRuNvWU0m3/TOx9CfvN60RPnk6iXGiRrDuNcrAWcpXamcsdmVgfkbNhc4j6P6XK68C8bncLeWJ5T7O+HOc5OKs1Fv271V1ZGgboJNKhT/7JK6XVpgjVy4ojeqUoBBlHNK9EqpJajdmZdlkrF2oNelnRgRFWlLNUSHhmuzh2Gq0lW/r/pKqnQnVCV5l6YkuW7tyo3I4Ilf5FRG77tRRGB0ucCroTNTp4tCkhWXzvy/V3hjdjZ4hp/1SkrDqSiA2I1GrB2N/3E8garpJWtz1MKITP9GOecsG9bUeRUBGLesPZMbJf4KwcuZrl051B1oOX6ccpBNxO0idx5CS63iL3SWCArUUtkx2dqW5DV5JtPNHL0xgIbdnISp4Te7S8EEM4mHF+C8NT4NA9JU/1DWjQ5Wfex6TtmNYi8LjC9SBEf0kNnMvAPTGSwMXHD3CZSXqDUqXWoB5moghOUKMWJ0RPhZ/gH3AF8u/zAghLApWUyd42vl36l9EjWbHggUF6v0gDCeQkh9TvVM9ERHX2vNap+kXU/gnsiKBuIYk2vvdmxUU3ShYpUapkuxpFVoTU/YqSuIU2SUqdbhgKVRXQTjqKwz+OszU3SpJTyDbA670h15JhUru+G8tPPXhQ9SmzwZEdlegGOYiVyb1aqt3wj6QRB6RoQV9dRQWk5pxgreQ6avBerbah41aey5qH70ZNSRaDaYAmmbukJFLW0f+js7hpMY8GQ7Xg1W8p+MSoY6arqBbkNdhaoZGi4Q/G7MRWcGnTYe4wu04NcTehSMXhnxAFTg1rE4VZJImKc3KVzXYZ+ZHzCVCi/DeJfcueqSsP9Dv5M4/jo3eztRy5Zr5Db+aYZ7i7v6mhAjVJDz3XjVurje1XpUUliVGtEF+fgi0EfTJq3O/JWSsVqlnmzqCFvRCilsPBtOmZO3POQx33p0XVo02V7ktnutPIRMEIuhgVLvM29Ibsrkpn2HduC5RrRMZ2SnozNPW6fkKuZBxJuBffY1q1A8WKZyVlDslCJ6TCIdmnSeGC1Umms5YNeYLcdNiiYIRRqqjXd7iYyltRO0uY7hvOgOyS/VNLsHFbTpLIav1yyC3UOTW9h1JIFYKMw00DdvhKhmeRCogGJ269Uy0x4ZKbYlLqyUZ+9eEjFqvIiDxbpLosFxR3YGVlSOF8OyqmlcnSxR7Fvj4qHdGd83g1MxF8P9ydKAYmAblo9daFxPb3wbiAGpKUwTdsOk7lBaWzsgI8yUBHFF6Nw3x2Di4Rynkbzy6ybpiRzxVpaOXUfW2ZkHKdKmhbJp9KJQVW0OP/dyl4yVKBpc/dulA10ZFGO6RvAi7DYTp61aUwOWLfJ+01xCoDIVxm1owDK7CmWJuxEi1Dt/wCjm27Xwgd3UoLWZOCz37puxr7kcUXO1bFsG8bmgdyJXrkFtHd11xurqCNycQ5GKTdg6unuJcKQUycQpjOmuM5zJXrOhQ4hgsyDPOEON4ZGReVrXdgPtU9pImTvn+7lTd/vXa/naygOSG2FcsXb2VmqBJX1rC/XHhlodtpxJqNCJPqd5t3lCs/Cc2xl+jcWj9XtIs6aHtFVhaYsPeVLPtIBUU91eWqvdQDulLC+OmRCVh5f87TLMW192Bb1AEknpTSlKqbGvM4FBYoSpbjNJkO0Gp5M0LKCp6yHPvhuuyNSRHL9V0KsTa7dt2uRtp1ZPnlNgvP5rEvfU6f7273X6t28aai7B+S3YJax6PidjVoOutmWLZV9g0XfcEE+5mAmTZcK4cnqh5aH6d49qafaQq8kTLrTtrgID3vUsntk6scfpOUVro+aXhXeg6UbnXgiboRw3j20bCTwxsknNfmrz2tSAv9YAZhK2R7DSuz1d628SB85RZrWbXICxudxO1Xsd5/BGvDLjd8WbPp+qQm4xX/SSnmkTNm1x073iUpWkM1LAM7GRXALJmBpHq6SxO95FIt1LDAzq9dI0cPamew3QNP6mdZLcIDk7mgWpJ176ezXNWkoCRYJ6M59Pjq/8S705gnebXhgTTC9L2zaPMQz91LdIZUiS546Gm7Zuw8Dr+v5EHWkduCNJ/8JzZlMQJcDMBvKqu4xMqXPThQO9gDjEQoSjBIdX5aJCfX0TefZjghWbNsyMrrzQ+ahTIpGpIeJ3TSbmBq2cUrnyEliCUui6x7XoOuctaBpO1hpeha7dEiYHJD6GnksQGyZkwdWuSoNs5bPK+nYz6dic6nz54tdzb8sAh95cO3IRZCaPhiqsHAgoebm0B5JmRoMc+b/ymRj1x18NsXsYv081jHguGJfNp56GPdtO/pxftrJnfyWQ08OPIJip5da5qhOaFyNjMPUq5JYB5aR1zCTxJ3RHIdLuff8UeEBwz8BwtjMnhuCruVOy6RPahkexzVjK0dOSKHorzi/r7nmll+t07zaTcTTXVJ8aaddNxo33tXvZiQprjC0/2IkdPrD0TmcZlmxrLdVxuCoWxbeAuqBKeQNZng/r5fcBUA2ozasNS179KsL2XRMxohLjSPV4nRF3MpIG2Og6/ywD1xAmMr8rRhF6GOI33VTVPlB+DQSatinYI9DI+U3mRaCjO3fvQYjHDpv5RWgz40woTS2523x+t3LMedTXDo8/4+Jc9Eh36Z3TgIUO3TuizZACVQvei0858O+H2D9FWwAB/3cr3a7WhpwaRhz0mHBgUSdhmbPdIyY6+e31WkaURyC79KoAuGp849L9pGkbF/4UH5x378UzYWvEB4RUXtrdT7eYl4YrKN/G++DQI/i1K8GHJx/FwI/R1QjsE9HT9bdH5DFrHXd4Rl2Qyio08k5a2m+SFrA84gGcKtn8cMwM8BLD9t5CeWQNbcS3Ppsb5M9hgW4wxZ2mxhWInTj2cvbU/27KwNnVpzfebGlLehfi5QKkxh+OIdM7X02PQiS2ln7Loj0TU+zbIvGWurL3am5emxq2vZd/c4SGIOy29dce21nWuZmiEHUmjJiZ2/r7RHU8R/5pBp32PAtmLqe3erFprJM41Jw370I3Wbxa+/YpCM2phdKyTDVdGS0t7X+miMJRfQejTdeRWA9+zlENdae33Rz2bh6ehl8wlN/Mv7JkM28JLdczm58vMWr3G0WmQWPO1vcdb3qy5MJ2tNcPLzrm5IdIShPFUY02HNLaxd21/RcUD0dq"
}

In [ ]:
#@title 0-3　載入資料
BASE_URL = "https://raw.githubusercontent.com/Po-yuLin/MedicalAICourse_DataPreprocessing/v1.0/data/"

def load_csv(name):
    """先從 GitHub 讀；失敗就改用內嵌的備援資料。全部以文字讀入，避免 pandas 偷偷幫你改資料。"""
    try:
        with urllib.request.urlopen(BASE_URL + name, timeout=10) as r:
            text, src = r.read().decode("utf-8"), "GitHub"
    except Exception:
        text, src = zlib.decompress(base64.b64decode(_EMBEDDED[name])).decode("utf-8"), "內嵌備援"
    print(f"  {name}：從「{src}」載入")
    return pd.read_csv(io.StringIO(text), dtype=str, keep_default_na=False)

df_dirty_small = load_csv("heart_dirty_small.csv")    # 40 位病人：給你手動清理
df_dirty_full  = load_csv("heart_dirty_full.csv")     # 全部病人：等一下讓程式自動清理
print(f"\n練習用小資料：{len(df_dirty_small)} 列；完整資料：{len(df_dirty_full)} 列")

In [ ]:
#@title 0-4　工具箱（檢查器、小型 MLP、前處理管線；不用展開）
import torch, torch.nn as nn
import matplotlib.pyplot as plt
torch.set_num_threads(2)
# ===== 資料字典與規則（檢查器與自動清理共用）=====
NUM_COLS = ["age", "resting_bp", "chol", "max_hr", "oldpeak", "ca"]          # 連續數值／計數
BIN_COLS = ["fbs", "exang"]                                                  # 0/1
CODE_COLS = ["rest_ecg", "slope"]                                            # 用數字編碼的類別
CAT_COLS = {"sex": ["male", "female"],
            "chest_pain": ["typical", "nontypical", "nonanginal", "asymptomatic"],
            "thal": ["normal", "fixed", "reversible"],
            "disease": ["yes", "no"]}
RANGES = {"age": (18, 100), "resting_bp": (70, 250), "chol": (100, 600),
          "max_hr": (60, 220), "oldpeak": (0, 8), "ca": (0, 3),
          "fbs": (0, 1), "exang": (0, 1), "rest_ecg": (0, 2), "slope": (1, 3)}
ALL_NUMERIC = NUM_COLS + BIN_COLS + CODE_COLS
MISSING_TOKENS = {"", "?", "na", "n/a", "nan", "null", "none", "unknown", "-", "--"}


def _is_missing(x):
    return pd.isna(x) or str(x).strip().lower() in MISSING_TOKENS


def check_cleaned(df, hint=1):
    """檢查清理後的表格。hint=1 只報數量；2 加上欄位；3 再加上 patient_id。"""
    issues = []   # (類型, 欄位, patient_id, 說明)
    d = df.copy()
    need = ["patient_id"] + ALL_NUMERIC + list(CAT_COLS)
    lost = [c for c in need if c not in d.columns]
    if lost:
        print(f"❌ 找不到欄位 {lost}。請不要更改標題列（第 1 列）的欄位名稱。")
        return False
    for i, row in d.iterrows():
        pid = row.get("patient_id", f"第{i+2}列")
        for col in ALL_NUMERIC:
            v = row[col]
            if _is_missing(v):
                issues.append(("缺值", col, pid, "空白或 ?、NA、unknown…"))
                continue
            try:
                x = float(str(v).strip())
            except ValueError:
                issues.append(("數字欄位混入文字", col, pid, f"「{v}」不是純數字"))
                continue
            lo, hi = RANGES[col]
            if col == "chol" and 0 < x < 30:
                issues.append(("單位疑似錯誤", col, pid, f"{x} 看起來是 mmol/L，要的是 mg/dL"))
            elif not (lo <= x <= hi):
                issues.append(("不可能的數值", col, pid, f"{x} 超出合理範圍 {lo}–{hi}"))
        for col, allowed in CAT_COLS.items():
            v = row[col]
            if _is_missing(v):
                issues.append(("缺值", col, pid, "空白或 NA…"))
            elif str(v) not in allowed:
                issues.append(("類別寫法不一致", col, pid, f"「{v}」不在 {allowed} 之中"))
    dup = d[d.duplicated("patient_id", keep="first")]
    for pid in dup["patient_id"]:
        issues.append(("重複列", "(整列)", pid, "同一位病人出現不只一次"))

    if not issues:
        print(f"✅ 全部通過！共 {len(d)} 列，沒有發現問題。")
        return True
    rep = pd.DataFrame(issues, columns=["類型", "欄位", "patient_id", "說明"])
    print(f"❌ 還有 {len(rep)} 個問題：")
    for kind, g in rep.groupby("類型"):
        print(f"  • {kind}：{len(g)} 個", end="")
        if hint >= 2:
            print(f"   欄位：{', '.join(sorted(g['欄位'].unique()))}", end="")
        print()
        if hint >= 3:
            for _, r in g.iterrows():
                print(f"      - {r['patient_id']} 的 {r['欄位']}：{r['說明']}")
    return False


def auto_clean(raw, fill=None):
    """用程式一次清完：同樣的規則，對 300 列跟 300 萬列都一樣快。
    fill：補缺值用的數字（中位數/眾數）。None＝由這批資料自己算；推論時要傳入「訓練時算好的」fill。"""
    d = raw.copy().astype(str)
    d = d.apply(lambda s: s.str.strip())
    d = d.replace({t: np.nan for t in ["", "?", "NA", "N/A", "na", "n/a", "null", "unknown", "-", "--"]})
    d = d.drop_duplicates(subset="patient_id", keep="first")           # 重複列
    for c in ALL_NUMERIC:                                              # 取出數字（去掉 mmHg、bpm…）
        d[c] = pd.to_numeric(d[c].str.extract(r"(-?\d+\.?\d*)")[0], errors="coerce")
    d.loc[d["chol"] < 30, "chol"] *= 38.67                             # mmol/L → mg/dL
    for c, (lo, hi) in RANGES.items():                                 # 不可能的值 → 視為缺值
        d.loc[(d[c] < lo) | (d[c] > hi), c] = np.nan
    fix = {                                                            # 類別寫法統一
        "sex": {"m": "male", "male": "male", "f": "female", "female": "female"},
        "chest_pain": {"typical": "typical", "typical angina": "typical",
                       "nontypical": "nontypical", "non-typical": "nontypical", "atypical": "nontypical",
                       "nonanginal": "nonanginal", "non-anginal": "nonanginal", "non anginal": "nonanginal",
                       "asymptomatic": "asymptomatic"},
        "thal": {"normal": "normal", "fixed": "fixed", "fixed defect": "fixed",
                 "reversible": "reversible", "reversable": "reversible", "reversible defect": "reversible"},
        "disease": {"yes": "yes", "y": "yes", "1": "yes", "no": "no", "n": "no", "0": "no"},
    }
    cats = [c for c in CAT_COLS if c in d.columns]
    for c in cats:
        d[c] = d[c].str.lower().map(fix[c])
    if fill is None:                                                   # 補缺值：數字用中位數、類別用眾數
        fill = {c: d[c].median() for c in ALL_NUMERIC}
        fill.update({c: d[c].mode()[0] for c in cats})
    for c in ALL_NUMERIC + cats:
        d[c] = d[c].fillna(fill[c])
    d["chol"] = d["chol"].round(0)
    d[ALL_NUMERIC] = d[ALL_NUMERIC].astype(float)
    d = d.reset_index(drop=True)
    d.attrs["fill"] = fill
    return d

# ===== 小型 MLP 與評估工具 =====
import torch, torch.nn as nn
from sklearn.metrics import roc_auc_score, recall_score, precision_score, accuracy_score
from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder

FEATURES_V1 = NUM_COLS + ["male"] + BIN_COLS + CODE_COLS      # 單元一、二使用：11 個全數字欄位
CATEGORIES = {"chest_pain": CAT_COLS["chest_pain"], "thal": CAT_COLS["thal"],
              "rest_ecg": [0.0, 1.0, 2.0], "slope": [1.0, 2.0, 3.0]}


def add_male(df):
    d = df.copy()
    d["male"] = (d["sex"] == "male").astype(float)
    return d


ROUGH = {"age": 100, "resting_bp": 200, "chol": 400, "max_hr": 200, "oldpeak": 6, "ca": 3, "rest_ecg": 2, "slope": 3}


def to_xy(df, features=FEATURES_V1, rough=True):
    """轉成模型吃的數字矩陣。rough=True：每個欄位除以一個固定的數字，縮到大約 0–1（單元 3 會解釋為什麼）。"""
    d = add_male(df)
    X = d[features].astype("float32").values
    if rough:
        for j, c in enumerate(features):
            if c in ROUGH:
                X[:, j] /= ROUGH[c]
    y = (d["disease"] == "yes").astype("float32").values
    return X, y


class MLP(nn.Module):
    """hidden=(3,)：輸入 → 3 個隱藏感知器（tanh，一點點非線性）→ 1 個輸出感知器。共 4 個感知器。"""
    def __init__(self, n_in, hidden=(3,)):
        super().__init__()
        layers, prev = [], n_in
        for h in hidden:
            layers += [nn.Linear(prev, h), nn.Tanh()]
            prev = h
        layers.append(nn.Linear(prev, 1))
        self.net = nn.Sequential(*layers)
    def forward(self, x):
        return self.net(x).squeeze(-1)


def train_mlp(X, y, epochs=300, lr=0.003, seed=0, pos_weight=None, X_val=None, y_val=None, hidden=(3,)):
    """整批資料一起訓練。回傳 (model, history)。history 裡有每個 epoch 的 loss。"""
    torch.manual_seed(seed)
    Xt, yt = torch.tensor(np.asarray(X), dtype=torch.float32), torch.tensor(np.asarray(y), dtype=torch.float32)
    model = MLP(Xt.shape[1], hidden)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    pw = None if pos_weight is None else torch.tensor(float(pos_weight))
    loss_fn = nn.BCEWithLogitsLoss(pos_weight=pw)
    hist = {"train": [], "val": []}
    for ep in range(epochs):
        model.train()
        opt.zero_grad()
        loss = loss_fn(model(Xt), yt)
        loss.backward()
        opt.step()
        hist["train"].append(loss.item())
        if X_val is not None:
            model.eval()
            with torch.no_grad():
                hist["val"].append(nn.BCEWithLogitsLoss()(model(torch.tensor(np.asarray(X_val), dtype=torch.float32)),
                                                          torch.tensor(np.asarray(y_val), dtype=torch.float32)).item())
    return model, hist


def predict_proba(model, X):
    model.eval()
    with torch.no_grad():
        return torch.sigmoid(model(torch.tensor(np.asarray(X), dtype=torch.float32))).numpy()


def evaluate(model, X, y, threshold=0.5):
    p = predict_proba(model, X)
    pred = (p >= threshold).astype(int)
    out = {"acc": accuracy_score(y, pred), "recall": recall_score(y, pred, zero_division=0),
           "precision": precision_score(y, pred, zero_division=0)}
    out["auc"] = roc_auc_score(y, p) if len(set(y)) > 1 and np.isfinite(p).all() else float("nan")
    return out

# ===== 前處理管線（單元三、四共用）=====
from sklearn.preprocessing import FunctionTransformer
from sklearn.model_selection import RepeatedStratifiedKFold, StratifiedKFold, train_test_split, GroupShuffleSplit
import matplotlib.pyplot as plt

PREP_NUM = NUM_COLS                       # 要標準化的連續數值
PREP_BIN = ["male"] + BIN_COLS            # 本來就是 0/1
PREP_CAT = ["chest_pain", "thal", "rest_ecg", "slope"]   # 類別


def _rough_scale(X):
    return np.asarray(X, dtype="float64") / np.array([ROUGH[c] for c in PREP_NUM])


def make_prep(encode="onehot", scale="standard"):
    """encode: 'none'（類別欄位只用數字碼，且不用文字欄位）/ 'int'（文字類別轉成整數）/ 'onehot'
    scale: 'none' 不縮放 / 'rough' 除以固定數字 / 'standard'（或 True）標準化"""
    if scale is True:
        scale = "standard"
    if scale is False:
        scale = "none"
    num_tf = {"none": "passthrough", "rough": FunctionTransformer(_rough_scale), "standard": StandardScaler()}[scale]
    steps = [("num", num_tf, PREP_NUM),
             ("bin", "passthrough", PREP_BIN)]
    if encode == "none":
        steps.append(("code", "passthrough", CODE_COLS))
    elif encode == "int":
        steps.append(("code", "passthrough", CODE_COLS))
        steps.append(("ord", OrdinalEncoder(categories=[CATEGORIES["chest_pain"], CATEGORIES["thal"]]),
                      ["chest_pain", "thal"]))
    elif encode == "onehot":
        steps.append(("onehot", OneHotEncoder(categories=[CATEGORIES[c] for c in PREP_CAT], sparse_output=False),
                      PREP_CAT))
    return ColumnTransformer(steps)


RESULTS = []          # 單元三所有實驗的成績，最後一起比較


def cv_eval(df, label, encode="onehot", scale="standard", augment=None, epochs=300, lr=0.003, repeats=3, record=True):
    """5 折交叉驗證 × repeats 次。前處理「只用訓練折」來 fit，再套用到驗證折。"""
    d = add_male(df).reset_index(drop=True)
    y = (d["disease"] == "yes").astype(float).values
    rows = []
    for k, (tr, va) in enumerate(RepeatedStratifiedKFold(n_splits=5, n_repeats=repeats, random_state=0).split(d, y)):
        prep = make_prep(encode, scale)
        Xtr = prep.fit_transform(d.iloc[tr]).astype("float32")
        Xva = prep.transform(d.iloc[va]).astype("float32")
        ytr = y[tr]
        if augment is not None:
            Xtr, ytr = augment(Xtr, ytr, k)
        model, _ = train_mlp(Xtr, ytr, epochs=epochs, lr=lr, seed=k)
        rows.append(evaluate(model, Xva, y[va]))
    r = pd.DataFrame(rows)
    out = dict(label=label, acc=r["acc"].mean(), acc_sd=r["acc"].std(), auc=r["auc"].mean(), auc_sd=r["auc"].std())
    if record:
        RESULTS.append(out)
    print(f"{label:<30} accuracy {out['acc']:.3f} ± {out['acc_sd']:.3f}   AUC {out['auc']:.3f} ± {out['auc_sd']:.3f}")
    return out


def noise_augment(copies=3, sd=0.1):
    """表格資料的擴增：把訓練資料複製幾份，連續數值加上一點雜訊。只能用在訓練集！"""
    def f(X, y, seed):
        r = np.random.default_rng(seed)
        mask = np.r_[np.ones(len(PREP_NUM)), np.zeros(X.shape[1] - len(PREP_NUM))].astype("float32")
        Xs, ys = [X], [y]
        for _ in range(copies):
            Xs.append(X + r.normal(0, sd, X.shape).astype("float32") * mask)
            ys.append(y)
        return np.vstack(Xs), np.concatenate(ys)
    return f

print("✅ 工具箱就緒（PyTorch", torch.__version__ + "）")

---
## 單元 1｜資料如何才能「餵進去」

AI 模型只吃**數字**。每一位病人是一列，每一欄是一個**特徵（feature）**，我們想預測的答案是**標籤（label）**：這裡是 `disease`（有沒有冠心病）。

### 資料字典

| 欄位 | 意義 | 合理範圍／標準寫法 |
|---|---|---|
| `patient_id` | 病人編號 | P001…；**每位病人只應出現一次** |
| `age` | 年齡（歲） | 18–100 |
| `sex` | 性別 | `male` / `female` |
| `chest_pain` | 胸痛類型 | `typical`（典型心絞痛）、`nontypical`（非典型）、`nonanginal`（非心絞痛）、`asymptomatic`（無症狀） |
| `resting_bp` | 休息時血壓（mmHg） | 70–250，**只要數字** |
| `chol` | 血清膽固醇（**mg/dL**） | 100–600，**只要數字** |
| `fbs` | 空腹血糖 > 120 mg/dL？ | `1`＝是、`0`＝否 |
| `rest_ecg` | 休息心電圖 | `0` 正常、`1` ST-T 異常、`2` 左心室肥厚 |
| `max_hr` | 運動測試最大心跳（bpm） | 60–220，**只要數字** |
| `exang` | 運動誘發心絞痛？ | `1`＝是、`0`＝否 |
| `oldpeak` | 運動後 ST 段下降（mm） | 0–8 |
| `slope` | 運動高峰 ST 段斜率 | `1` 上升、`2` 平、`3` 下降 |
| `ca` | 透視下有鈣化的主要血管數 | 0–3 |
| `thal` | 鉈（thallium）心肌灌注掃描 | `normal` / `fixed` / `reversible` |
| `disease` | **標籤**：有冠心病嗎？ | `yes` / `no` |

> 你不需要懂這些指標的臨床意義，只需要知道**每一欄「應該長什麼樣子」**。

In [ ]:
#@title 1-1　👀 先看看：直接把髒資料餵給模型會怎樣？
print("髒資料的前 8 列：")
display(df_dirty_small.head(8))

# 硬把每個欄位轉成數字；轉不了的（空白、?、NA、"120 mmHg"…）會變成 NaN（Not a Number）
d_bad = df_dirty_small.copy()
for c in ALL_NUMERIC:
    d_bad[c] = pd.to_numeric(d_bad[c], errors="coerce")
print("\n硬轉成數字後，這些欄位出現了 NaN：")
print(d_bad[ALL_NUMERIC].isna().sum()[lambda s: s > 0].to_string())

X_bad = add_male(d_bad)[FEATURES_V1].values.astype("float32")
y_bad = df_dirty_small["disease"].str.strip().str.lower().isin(["yes", "y", "1"]).values.astype("float32")
_, h_bad = train_mlp(X_bad, y_bad, epochs=10)
print("\n訓練 10 個 epoch，每一步的 loss（模型的『錯誤程度』）：")
print([round(v, 3) for v in h_bad["train"]])
print("\n👉 loss 變成 nan：模型一碰到 NaN，整個訓練就壞掉了，而且不會告訴你為什麼。")

In [ ]:
#@title 1-2　⚠️ 更可怕的是「不會報錯」的錯誤
print("sex 欄位實際出現的寫法：")
print(df_dirty_small["sex"].value_counts().to_string())

naive_male = (df_dirty_small["sex"] == "male").sum()
real_male  = df_dirty_small["sex"].str.strip().str.lower().isin(["m", "male"]).sum()
print(f"\n如果程式寫「sex == 'male'」，會數到 {naive_male} 位男性；")
print(f"但實際上男性有 {real_male} 位。有 {real_male - naive_male} 位被默默當成女性，程式完全沒有報錯。")

In [ ]:
#@title 1-3　建立練習用的 Google 試算表（第一次會跳出授權視窗）
MODE = "sheets"          # "sheets"＝Google 試算表（建議）；"csv"＝下載 CSV 編輯後再上傳（備案，見下方說明）
SHEET_TITLE = "heart_dirty_practice"
CLEANED_CSV = os.path.join(WORKDIR, "heart_cleaned_small.csv")      # csv 模式：請把清理好的檔案用這個名字存到 Drive 資料夾

df_dirty_small.to_csv(os.path.join(WORKDIR, "heart_dirty_small.csv"), index=False)   # 備案用：Drive 資料夾裡也放一份

def _to_cell(v):
    """數字寫成數字，其他維持文字（試算表裡看起來才像真的資料）。"""
    try:
        f = float(v)
        return int(f) if (f.is_integer() and "." not in v) else f
    except (ValueError, TypeError):
        return v

if MODE == "sheets" and not IN_COLAB:
    print("（不在 Colab 環境，自動改用 csv 模式）"); MODE = "csv"

if MODE == "sheets":
    from google.colab import auth
    auth.authenticate_user()
    import gspread
    from google.auth import default
    gc = gspread.authorize(default()[0])
    try:
        sh = gc.open(SHEET_TITLE)
        print("✅ 找到先前建立的試算表，你之前的修改都還在。")
    except gspread.SpreadsheetNotFound:
        sh = gc.create(SHEET_TITLE)
        rows = [list(df_dirty_small.columns)] + [[_to_cell(v) for v in r] for r in df_dirty_small.values.tolist()]
        sh.sheet1.update(values=rows, range_name="A1")
        print("✅ 已建立新的試算表。")
    print("\n👉 請點開這個連結，開始清理（也可以在 Google 雲端硬碟找到它）：")
    print("  ", sh.url)
    print("\n（想整個重來：把這個試算表丟進垃圾桶，再重新執行這一格）")
else:
    print("csv 模式：")
    print(f"  1. 到 Google 雲端硬碟 → AI_preprocessing_class 資料夾 → 找到 heart_dirty_small.csv")
    print( "  2. 用 Google 試算表（或 Excel）開啟，修改")
    print( "  3. 另存／下載成 CSV，檔名改成 heart_cleaned_small.csv，上傳回同一個資料夾")

### 👉 你的任務（約 15 分鐘，兩人一組）

打開試算表，把這 40 位病人的資料**清理到可以餵給模型**。每種問題各有幾個，找出來並修好：

| 問題類型 | 怎麼辨認 | 可以怎麼處理 |
|---|---|---|
| 缺值 | 空白、`?`、`NA`、`unknown`、`N/A`… | 補一個合理的值（例如該欄中位數），或刪掉整列 |
| 不可能的數值 | 超出上面表格的合理範圍 | 能判斷是打錯（多打一個 0？）就改；不能判斷就當缺值 |
| 單位不一致 | 膽固醇應該是 100–600；出現 5.46 可能是 mmol/L | mg/dL ＝ mmol/L × 38.67 |
| 數字欄位混入文字 | `110 mmHg`、`143bpm` | 只留下數字 |
| 類別寫法不一 | `M`/`Male`、`ASYMPTOMATIC`、`reversable`、`1`/`Y`… | 改成上面表格的**標準寫法**（小寫、沒有多餘空格） |
| 重複列 | 同一個 `patient_id` 出現兩次 | 只留一列 |

**小技巧**：`Ctrl+H`（尋找與取代）、「資料 → 資料清理 → 移除重複項目」、點欄位標題排序，會讓你快很多。
**規則**：不要改標題列（第 1 列）的欄位名稱。
**想一想**：遇到「看起來很誇張」的數值，你要怎麼判斷它是**打錯**還是**真的**？刪掉它，有沒有代價？

修完之後，執行下面兩格：先檢查，再訓練。檢查會告訴你還剩什麼問題；卡住就把 `HINT` 調大一點。

**🎓 老師備註（單元 1）**
- 40 列裡共注入：缺值 5＋原本就缺 2、不可能值 4、單位錯誤 3、文字混入 4、類別寫法 11、重複 2。完整答案：`teacher/answer_key_small.csv`。
- **陷阱題（刻意保留）**：`P153` 的 chol = 564、`P092` 的 oldpeak = 6.2，都是**真實**極端值，不是錯誤。檢查器不會抓它們。若學生刪除或改動，正好討論「由有臨床判斷力的人決定」。
- 原本就缺值的是 `P088`（thal）與 `P167`（ca）：不是我們造的，是資料集本來就有。
- 學生卡住或你想直接示範：執行下一格「老師捷徑」，等於一位完美的學生。

**🎓 老師備註：小資料集（40 位）的注入錯誤完整答案**（另外 P088 的 thal、P167 的 ca 是資料集本來就缺）

| patient_id | 欄位 | 類型 | 髒值 | 原本的值 |
|---|---|---|---|---|
| P111 | chol | 缺值 | `` | `307` |
| P268 | ca | 缺值 | `?` | `1` |
| P224 | thal | 缺值 | `` | `reversible` |
| P135 | resting_bp | 缺值 | `unknown` | `122` |
| P145 | oldpeak | 缺值 | `` | `0.6` |
| P110 | age | 不可能值 | `520` | `39` |
| P247 | resting_bp | 不可能值 | `0` | `100` |
| P079 | chol | 不可能值 | `0` | `245` |
| P189 | max_hr | 不可能值 | `1950` | `195` |
| P059 | chol | 單位錯誤 | `7.06` | `273` |
| P021 | chol | 單位錯誤 | `5.46` | `211` |
| P078 | chol | 單位錯誤 | `7.96` | `308` |
| P109 | resting_bp | 數字欄位混入文字 | `120 mmHg` | `120` |
| P058 | resting_bp | 數字欄位混入文字 | `110 mmHg` | `110` |
| P145 | chol | 數字欄位混入文字 | `240 mg/dL` | `240` |
| P123 | max_hr | 數字欄位混入文字 | `143bpm` | `143` |
| P135 | sex | 類別寫法 | `female ` | `female` |
| P167 | sex | 類別寫法 | `MALE` | `male` |
| P110 | sex | 類別寫法 | `m` | `male` |
| P009 | chest_pain | 類別寫法 | `ASYMPTOMATIC` | `asymptomatic` |
| P110 | chest_pain | 類別寫法 | `asymptomatic ` | `asymptomatic` |
| P153 | chest_pain | 類別寫法 | `Nonanginal` | `nonanginal` |
| P111 | thal | 類別寫法 | `reversible defect` | `reversible` |
| P236 | thal | 類別寫法 | `NORMAL` | `normal` |
| P247 | disease | 類別寫法 | `Yes` | `yes` |
| P132 | disease | 類別寫法 | `0` | `no` |
| P109 | disease | 類別寫法 | `y` | `yes` |
| P210 | (整列) | 重複列 | `duplicate row` | `` |
| P128 | (整列) | 重複列 | `duplicate row` | `` |

In [ ]:
#@title [老師] 捷徑：模擬一位把資料清得很完美的學生（學生版沒有這一格）
_perfect = auto_clean(df_dirty_small)
if MODE == "csv":
    _perfect.to_csv(CLEANED_CSV, index=False)
else:
    sh.sheet1.clear()
    sh.sheet1.update(values=[list(_perfect.columns)] + _perfect.values.tolist(), range_name="A1")
print("✅ 已用程式清理並寫回（相當於一位完美學生）。")

In [ ]:
#@title 1-4　讀回你清理的結果並檢查（可以重複執行，直到全部通過）
HINT = 1     # 1：只告訴你還有幾類問題　2：再告訴你是哪幾個欄位　3：連 patient_id 都告訴你

def read_student_table():
    if MODE == "sheets":
        rows = sh.sheet1.get_all_values()
        d = pd.DataFrame(rows[1:], columns=rows[0])
    else:
        d = pd.read_csv(CLEANED_CSV, dtype=str, keep_default_na=False)
    return d[d.apply(lambda r: "".join(map(str, r)).strip() != "", axis=1)].reset_index(drop=True)   # 去掉全空白的列

df_student = read_student_table()
print(f"讀到 {len(df_student)} 列（原本 {len(df_dirty_small)} 列）")
if len(df_student) < len(df_dirty_small) - 6:
    print("⚠️ 你刪掉的列比較多。刪掉的每一列都是少一位病人的資訊，確定每一列都真的該刪嗎？")
all_ok = check_cleaned(df_student, hint=HINT)

In [ ]:
#@title 1-5　用你清好的資料，訓練一個超小的神經網路（輸入 → 3 個感知器 → 1 個感知器，共 4 個）
def train_on_table(df, epochs=300):
    try:
        X, y = to_xy(df)
    except Exception as e:
        print("❌ 資料還沒辦法餵進模型。Python 的錯誤訊息是：\n  ", e)
        print("   → 回到 1-4 看檢查結果；通常是數字欄位裡還有文字或空白。")
        return None
    if np.isnan(X).any():
        print("⚠️ 資料裡還有 NaN，loss 會變成 nan。")
    model, hist = train_mlp(X, y, epochs=epochs)
    plt.figure(figsize=(5.5, 3)); plt.plot(hist["train"])
    plt.xlabel("epoch"); plt.ylabel("training loss"); plt.title("Training loss"); plt.tight_layout(); plt.show()
    acc = evaluate(model, X, y)["acc"]
    print(f"訓練 {epochs} 個 epoch：loss {hist['train'][0]:.3f} → {hist['train'][-1]:.3f}；"
          f"在這 {len(y)} 位病人上的準確率 {acc:.1%}")
    print("（to_xy 會先把每個欄位縮到大約 0–1，小網路才學得起來；單元 3 會解釋為什麼。）")
    return model, hist

result_student = train_on_table(df_student)

### 同一件事，讓程式來做

你剛剛花了十幾分鐘，只清理了 **40 列**。真實世界的資料動輒幾萬、幾百萬列，不可能手動。
但請注意：程式做的，其實就是你剛剛做的每一個判斷，只是把它們**寫成規則**——所以**你得先懂資料有哪些問題**，程式才寫得出來。

In [ ]:
#@title 1-6　用一格程式清理「全部」資料
t0 = time.time()
df_clean = auto_clean(df_dirty_full)
elapsed = time.time() - t0
print(f"原始 {len(df_dirty_full)} 列 → 清理後 {len(df_clean)} 列（移除 {len(df_dirty_full) - len(df_clean)} 列重複），只花了 {elapsed:.2f} 秒")
check_cleaned(df_clean)
df_clean.to_csv(os.path.join(WORKDIR, "heart_clean_full.csv"), index=False)
display(df_clean.head())
result_full = train_on_table(df_clean)

### 先打個問號 ❓

剛剛顯示的準確率，是模型在**它自己學過的資料**上算出來的。
**這個數字可信嗎？** 就像學生拿「考古題」當考試成績——這就是單元 2 要談的事。

**🎓 老師備註**：自動清理用「中位數／眾數」補缺值，且是用**全部資料**算的，這本身就是輕微的資訊洩漏；單元 4 會示範正確做法：補值用的數字也要「只用訓練資料算、存起來、推論時重複使用」。

---
## 單元 2｜資料怎麼分配

模型的目的，是在**沒看過的新病人**身上也準。所以我們不能用「它學過的資料」來評分。
把資料分成三份，就像準備考試：

| 資料 | 像是 | 用途 | 可以用幾次 |
|---|---|---|---|
| **訓練集 train** | 課本與考古題 | 模型從這裡學習 | 無限次 |
| **驗證集 validation** | 模擬考 | 比較不同做法、調整設定 | 很多次（但用越多，越會偏向它） |
| **測試集 test** | **正式考試** | 最後給模型一個誠實的成績 | **只有一次** |

In [ ]:
#@title 2-1　隨機切成「訓練 80％／測試 20％」，換 5 個隨機種子看看
if "df_clean" not in globals():                 # 保險：沒跑單元 1 也能繼續
    df_clean = auto_clean(df_dirty_full)
X_all, y_all = to_xy(df_clean)

accs = []
for seed in range(5):
    Xtr, Xte, ytr, yte = train_test_split(X_all, y_all, test_size=0.2, random_state=seed)
    model, _ = train_mlp(Xtr, ytr)
    acc = evaluate(model, Xte, yte)["acc"]
    accs.append(acc)
    print(f"random_state={seed}：測試集準確率 {acc:.1%}　（測試集只有 {len(yte)} 位病人）")

plt.figure(figsize=(5, 2.8)); plt.plot(range(5), accs, "o-"); plt.ylim(0.4, 1)
plt.xlabel("random_state (how the data was split)"); plt.ylabel("test accuracy"); plt.tight_layout(); plt.show()
print(f"\n👉 同一個模型、同一份資料，只是『切法』不同，準確率就從 {min(accs):.0%} 跳到 {max(accs):.0%}。")
print("   測試集只有 61 個人，多抽到幾個難判斷的病人，成績就差很多。要小心『一次切分的結果』。")

In [ ]:
#@title 2-2　切成 訓練／驗證／測試 三份，並把「測試集」鎖起來
TEST_SIZE, VAL_SIZE, SEED = 0.20, 0.20, 42        # 👉 可以改改看

is_yes = df_clean["disease"] == "yes"
trainval_df, test_df = train_test_split(df_clean, test_size=TEST_SIZE, random_state=SEED, stratify=is_yes)
train_df, val_df = train_test_split(trainval_df, test_size=VAL_SIZE / (1 - TEST_SIZE), random_state=SEED,
                                    stratify=trainval_df["disease"] == "yes")      # stratify：讓每份的『有病比例』差不多

summary = pd.DataFrame({
    "人數": [len(train_df), len(val_df), len(test_df)],
    "有病比例": [(d["disease"] == "yes").mean() for d in (train_df, val_df, test_df)],
}, index=["訓練集 train", "驗證集 validation", "測試集 test"])
display(summary.style.format({"有病比例": "{:.1%}"}))

test_df.to_csv(os.path.join(WORKDIR, "heart_test_locked.csv"), index=False)
del test_df
print("🔒 測試集已存到 Drive，並從記憶體刪除。單元 4 才會打開它，而且只打開一次。")

Xtr, ytr = to_xy(train_df); Xva, yva = to_xy(val_df)
model, _ = train_mlp(Xtr, ytr)
print(f"\n用訓練集學習，在驗證集上的準確率：{evaluate(model, Xva, yva)['acc']:.1%}")

In [ ]:
#@title 2-3　K 折交叉驗證：資料太少時，讓每一位病人都輪流當一次「驗證」
K = 5
Xtv, ytv = to_xy(trainval_df)
member = np.zeros((K, len(trainval_df)))                 # 0＝這個人這一折在訓練，1＝在驗證
fold_acc = []
for k, (tr, va) in enumerate(StratifiedKFold(n_splits=K, shuffle=True, random_state=SEED).split(Xtv, ytv)):
    member[k, va] = 1
    model, _ = train_mlp(Xtv[tr], ytv[tr], seed=k)
    fold_acc.append(evaluate(model, Xtv[va], ytv[va])["acc"])
    print(f"第 {k+1} 折：訓練 {len(tr)} 人、驗證 {len(va)} 人 → 驗證準確率 {fold_acc[-1]:.1%}")
print(f"\n平均 {np.mean(fold_acc):.1%} ± {np.std(fold_acc):.1%}（± 是 5 折之間的標準差）")

from matplotlib.colors import ListedColormap
order = np.argsort(member.argmax(axis=0))
plt.figure(figsize=(8, 2.2))
plt.imshow(member[:, order], aspect="auto", cmap=ListedColormap(["#9ecae1", "#de2d26"]), interpolation="nearest")
plt.yticks(range(K), [f"fold {i+1}" for i in range(K)]); plt.xlabel("patients (blue = train, red = validation)")
plt.tight_layout(); plt.show()
print("👉 報告成績時，請報『平均 ± 標準差』，不要只報某一次切分的最好結果。")

### ⚠️ 最容易犯、也最難發現的錯：資料洩漏（data leakage）

想像每位病人做了**好幾次檢查**（或一段 EEG 被切成好幾個片段）。如果你把這些「列」隨機打散去切分，**同一位病人就會同時出現在訓練集和測試集**。
模型不需要真的學會判斷疾病，只要「認得這個人」就能答對。

In [ ]:
#@title 2-4　洩漏示範：同一位病人同時出現在訓練與測試
rng = np.random.default_rng(0)

def make_multi_visit(d, visits=5, jitter=0.03):
    """假設每位病人檢查了 5 次，每次的數值有一點點測量誤差。"""
    out = []
    for _ in range(visits):
        e = d.copy()
        for c in ["age", "resting_bp", "chol", "max_hr", "oldpeak"]:
            e[c] = e[c] * rng.normal(1, jitter, len(e))
        out.append(e)
    return pd.concat(out, ignore_index=True)

multi = make_multi_visit(trainval_df)
Xm, ym = to_xy(multi); groups = multi["patient_id"].values
print(f"{len(trainval_df)} 位病人 × 5 次檢查 = {len(multi)} 列")

def fit_eval(tr, te, seed):
    sc = StandardScaler().fit(Xm[tr])        # 標準化：單元 3 會細講，這裡先用，讓模型學得更好
    model, _ = train_mlp(sc.transform(Xm[tr]), ym[tr], lr=0.01, seed=seed, hidden=(64, 32))
    # 這裡刻意用「大」的網路：越大的模型越會『背答案』，洩漏的傷害就越大
    return evaluate(model, sc.transform(Xm[te]), ym[te])["acc"]

acc_row, acc_pat = [], []
for seed in range(4):
    tr, te = train_test_split(np.arange(len(multi)), test_size=0.2, random_state=seed)              # ❌ 以「列」切分
    acc_row.append(fit_eval(tr, te, seed))
    tr, te = next(GroupShuffleSplit(1, test_size=0.2, random_state=seed).split(Xm, ym, groups))     # ✅ 以「病人」切分
    acc_pat.append(fit_eval(tr, te, seed))

print(f"\n❌ 以『列』隨機切分（同一位病人跨訓練/測試）：測試準確率 {np.mean(acc_row):.1%}")
print(f"✅ 以『病人』切分（每位病人只在一邊）　　　：測試準確率 {np.mean(acc_pat):.1%}")
plt.figure(figsize=(4, 2.8)); plt.bar(["split by row\n(leaky)", "split by patient\n(correct)"], [np.mean(acc_row), np.mean(acc_pat)],
                                      color=["#de2d26", "#3182bd"]); plt.ylim(0.5, 1); plt.ylabel("test accuracy"); plt.tight_layout(); plt.show()
print("\n👉 好到不真實的成績，要先懷疑『有沒有洩漏』，再開心。")

**🎓 老師備註（單元 2）**
- 2-1 的準確率會明顯跳動（小網路、只做粗略縮放，準確率大約 70–90%）：這是刻意的，單元 3 會示範更好的縮放。
- 2-4 的差距大約是 99％ vs 78％。可以問：「如果這是一篇論文的結果，你會怎麼發現？」→ 看切分單位是病人還是影像／切片。
- 也可提：EEG 的 epoch、MRI 的切片、同一病人的多次住院，都是同樣的結構。

---
## 單元 3｜前處理如何影響訓練

接下來每個實驗都用**同一批資料、同一個小網路**，只改「前處理」這一件事。
為了不被運氣影響，每個實驗都用 5 折交叉驗證重複 3 次（共 15 次訓練），報告**平均 ± 標準差**。
（只用訓練＋驗證資料；測試集還鎖著。）

我們用的網路超小：**輸入 → 3 個感知器 → 1 個感知器，總共 4 個**（約 70 個參數）。資料只有 300 筆，網路大了只會背答案。

In [ ]:
#@title 3-1　標準化：把每個欄位拉到同一把尺上
cols = ["age", "resting_bp", "chol", "max_hr", "oldpeak", "ca"]
print("縮放之前，各欄位的『大小』差很多：")
display(trainval_df[cols].describe().loc[["mean", "std", "min", "max"]].round(1))
print("膽固醇動輒 200–300，oldpeak 只有 0–6。數值大的欄位會『壓過』數值小的欄位。")
print("（單元 1、2 的 to_xy 其實偷偷做了『粗略版』：每欄除以一個固定的數字，縮到大約 0–1。現在來比較三種做法。）\n")

def loss_curves(scale):
    prep = make_prep("onehot", scale)
    Xtr = prep.fit_transform(add_male(train_df)).astype("float32")       # 只用「訓練集」算出縮放需要的數字
    Xva = prep.transform(add_male(val_df)).astype("float32")             # 驗證集只能「套用」，不能參與計算
    ytr = (train_df["disease"] == "yes").values.astype("float32"); yva = (val_df["disease"] == "yes").values.astype("float32")
    return train_mlp(Xtr, ytr, X_val=Xva, y_val=yva)[1]

curves = {"no scaling": loss_curves("none"), "rough (divide by fixed number)": loss_curves("rough"), "standardized": loss_curves("standard")}
fig, ax = plt.subplots(1, 2, figsize=(10, 3), sharey=True)
for a_, key, title in zip(ax, ["train", "val"], ["training loss", "validation loss"]):
    for name, h in curves.items():
        a_.plot(h[key], label=name)
    a_.set_title(title); a_.set_xlabel("epoch"); a_.legend(fontsize=8)
plt.tight_layout(); plt.show()

RESULTS.clear()
print("五折交叉驗證 × 3 次，三種做法的成績：")
_ = cv_eval(trainval_df, "S0. no scaling", encode="onehot", scale="none")
_ = cv_eval(trainval_df, "S1. rough scaling", encode="onehot", scale="rough")
res_std = cv_eval(trainval_df, "S2. standardize", encode="onehot", scale="standard")
print("\n👉 不縮放：成績明顯較低，而且很不穩定（± 很大）；縮放之後又好又穩。標準化＝減平均、除以標準差，讓每欄平均 0、標準差 1。")

**標準化的重點**：`(數值 − 訓練集的平均) ÷ 訓練集的標準差`。注意這裡的「平均」和「標準差」是**從訓練集算出來**，之後套用到驗證集、測試集、新病人身上。

**🎓 老師備註**：預期 S0 約 70％（± 約 14，很不穩定）、S1 約 82％、S2 約 83％。看 loss 曲線：S0 下降慢又抖。S1 vs S2 的差距不大，可以說明「粗略縮放也能用，標準化是更標準、更穩的版本」。

In [ ]:
#@title 3-2　類別資料：整數編碼 vs One-hot（之後的實驗都使用標準化）
print("原始表格：類別是文字")
display(trainval_df[["patient_id", "chest_pain", "thal"]].head(4))

print("\nOne-hot：每個類別變成一欄 0/1")
oh = pd.get_dummies(trainval_df[["chest_pain"]]).astype(int)
display(pd.concat([trainval_df[["patient_id", "chest_pain"]], oh], axis=1).head(4))

print("\n整數編碼會是 typical=0, nontypical=1, nonanginal=2, asymptomatic=3。")
print("模型會以為 asymptomatic(3) 是 typical(0) 的『三倍』、nonanginal 在『中間』，但這個順序是我們隨便排的。")
print("\n各類型實際的有病比例：")
display((trainval_df.groupby("chest_pain")["disease"].apply(lambda s: (s == "yes").mean())
         .round(2).to_frame("有病比例")))

print("\n三種做法的比較：")
_ = cv_eval(trainval_df, "A. numbers only (no text cols)", encode="none")
_ = cv_eval(trainval_df, "B. integer codes", encode="int")
_ = cv_eval(trainval_df, "C. one-hot", encode="onehot")

**怎麼讀這個結果？** 加進胸痛類型與 thal 之後，成績有沒有變好（A → B/C）？One-hot 比整數編碼好多少？看看差距有沒有大過 ± 的範圍。
資料很少時，**小的差異可能只是運氣**，這就是為什麼我們要看平均 ± 標準差。

**🎓 老師備註**：A → B/C 有進步；B vs C 常在雜訊範圍內。可以誠實地說：「在這個小網路與小資料上，one-hot 的好處不大；它真正重要的場景是類別很多（例如 ICD 碼）或線性模型。但**整數編碼會引入不存在的順序，是概念上的錯誤**。」

In [ ]:
#@title 3-3　⚠️ 洩漏：「先標準化全部資料、再切分」 vs 「只用訓練集的平均與標準差」
d = add_male(trainval_df).reset_index(drop=True)
y = (d["disease"] == "yes").astype(float).values
X_leaky = make_prep("onehot", True).fit_transform(d).astype("float32")       # ❌ 平均/標準差用了全部資料，包含之後要拿來『考試』的人
rows = []
for k, (tr, va) in enumerate(RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=0).split(d, y)):
    model, _ = train_mlp(X_leaky[tr], y[tr], seed=k)
    rows.append(evaluate(model, X_leaky[va], y[va]))
r = pd.DataFrame(rows)
proper = res_std
print(f"❌ 先標準化再切分：accuracy {r['acc'].mean():.3f}   AUC {r['auc'].mean():.3f}")
print(f"✅ 只用訓練集　　：accuracy {proper['acc']:.3f}   AUC {proper['auc']:.3f}")
print("\n這個資料集上兩者幾乎一樣：因為只是一點點平均值與標準差的資訊『滲』過去。")
print("但原則不能破：任何『從資料算出來的東西』（平均、標準差、補缺值用的中位數、挑特徵）都只能用訓練資料算。")

In [ ]:
#@title 3-4　（加碼，時間夠再跑）洩漏有多可怕？用「完全隨機」的資料看看
rng = np.random.default_rng(0)
n = len(trainval_df)
X_noise = rng.normal(size=(n, 300)).astype("float32")      # 300 個純雜訊的「基因」
y_noise = rng.integers(0, 2, n).astype("float32")          # 隨機決定有病或沒病：根本不可能預測

def top_k(X, y, k=10):
    Xc = (X - X.mean(0)) / X.std(0); yc = (y - y.mean()) / y.std()
    return np.argsort(-np.abs((Xc * yc[:, None]).mean(0)))[:k]

leaky, proper_ = [], []
for k, (tr, va) in enumerate(RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=0).split(X_noise, y_noise)):
    sel = top_k(X_noise, y_noise)                      # ❌ 用「全部」資料的答案來挑特徵
    m, _ = train_mlp(X_noise[tr][:, sel], y_noise[tr], seed=k, lr=0.01); leaky.append(evaluate(m, X_noise[va][:, sel], y_noise[va])["acc"])
    sel = top_k(X_noise[tr], y_noise[tr])              # ✅ 只用訓練折挑特徵
    m, _ = train_mlp(X_noise[tr][:, sel], y_noise[tr], seed=k, lr=0.01); proper_.append(evaluate(m, X_noise[va][:, sel], y_noise[va])["acc"])
print(f"❌ 先用全部資料挑特徵：準確率 {np.mean(leaky):.1%}")
print(f"✅ 只用訓練資料挑特徵：準確率 {np.mean(proper_):.1%}　（隨機資料，理論上就是 50％）")
print("\n👉 資料裡完全沒有訊號，洩漏卻讓模型看起來『有預測力』。基因體學、影像組學（特徵多、人數少）的論文最容易中招。")

In [ ]:
#@title 3-5　資料擴增：表格資料加雜訊
_ = cv_eval(trainval_df, "E. + noise augmentation (x3)", encode="onehot", scale="standard", augment=noise_augment(3, 0.1))
print("\n👉 在這個小資料集上，擴增幾乎沒有幫助（差異在 ± 範圍內）。擴增不是萬靈丹，一樣要用驗證的方式檢查有沒有用。")
print("   而且：只能擴增『訓練集』。如果驗證集也擴增，同一個病人的變形版本就會跨過兩邊，又是洩漏。")

In [ ]:
#@title 3-6　影像資料的擴增（純示範，不訓練）：哪些變形在醫學上合理？
try:
    from skimage.data import shepp_logan_phantom
    from skimage.transform import resize, rotate
    head = resize(shepp_logan_phantom(), (200, 200))               # 標準的「頭部切面」測試影像（模擬，非真實病人）
except Exception:
    yy0, xx0 = np.mgrid[:200, :200]; head = (((xx0 - 100) / 80) ** 2 + ((yy0 - 100) / 95) ** 2 < 1) * 0.6
    rotate = lambda im, a: im
yy, xx = np.mgrid[:200, :200]
img = head.copy(); img[((xx - 60) ** 2 + (yy - 95) ** 2) < 14 ** 2] = 1.0       # 在影像「左側」放一個假的病灶
panels = [("original", img), ("flip left-right", img[:, ::-1]), ("rotate 10 deg", rotate(img, 10)),
          ("brightness x1.3", np.clip(img * 1.3, 0, 1)),
          ("gaussian noise", np.clip(img + np.random.default_rng(0).normal(0, 0.08, img.shape), 0, 1))]
fig, ax = plt.subplots(1, 5, figsize=(13, 3))
for a, (t, im) in zip(ax, panels):
    a.imshow(im, cmap="gray", vmin=0, vmax=1); a.set_title(t); a.axis("off")
plt.tight_layout(); plt.show()
print("左右翻轉之後，病灶跑到另一側。如果你的任務跟『哪一側』有關（例如左/右側中風），這種擴增會直接把標籤弄錯。")
print("旋轉一點點、調亮度、加雜訊，通常合理（模擬病人擺位、不同掃描儀）。什麼擴增合理，要由懂臨床的人決定。")

In [ ]:
#@title 3-7　資料不平衡：罕見疾病只占一小部分
pos = trainval_df[trainval_df["disease"] == "yes"]; neg = trainval_df[trainval_df["disease"] == "no"]
n_pos = int(round(0.15 * len(neg) / 0.85))
imb_df = pd.concat([neg, pos.sample(n_pos, random_state=1)]).sample(frac=1, random_state=1).reset_index(drop=True)
yi = (imb_df["disease"] == "yes").astype(float).values
print(f"把有病的人抽樣到約 15％：共 {len(imb_df)} 人，其中有病 {int(yi.sum())} 人（{yi.mean():.0%}）")

def oof_proba(strategy, repeats=3):
    """交叉驗證，把每一位病人『沒被訓練到的那一折』的預測機率收集起來。"""
    P = np.zeros(len(imb_df))
    for rep in range(repeats):
        for j, (tr, va) in enumerate(StratifiedKFold(5, shuffle=True, random_state=rep).split(imb_df, yi)):
            prep = make_prep("onehot", True)
            Xtr = prep.fit_transform(add_male(imb_df.iloc[tr])).astype("float32"); Xva = prep.transform(add_male(imb_df.iloc[va])).astype("float32")
            ytr, pw = yi[tr], None
            if strategy == "class weight":                       # 讓「錯過一個病人」的懲罰變重
                pw = (ytr == 0).sum() / (ytr == 1).sum()
            if strategy == "oversample":                         # 把少數類別（有病）重複抽樣補到跟多數一樣多
                idx = np.where(ytr == 1)[0]
                extra = np.random.default_rng(j).choice(idx, (ytr == 0).sum() - len(idx), replace=True)
                Xtr, ytr = np.vstack([Xtr, Xtr[extra]]), np.r_[ytr, ytr[extra]]
            model, _ = train_mlp(Xtr, ytr, seed=j, pos_weight=pw)
            P[va] += predict_proba(model, Xva) / repeats
    return P

rows = [{"strategy": "always predict 'no disease'", "accuracy": 1 - yi.mean(), "recall": 0.0, "precision": float("nan"), "AUC": 0.5}]
for s in ["plain", "class weight", "oversample"]:
    P = oof_proba(s); pred = (P >= 0.5).astype(int)
    rows.append({"strategy": s, "accuracy": accuracy_score(yi, pred), "recall": recall_score(yi, pred),
                 "precision": precision_score(yi, pred, zero_division=0), "AUC": roc_auc_score(yi, P)})
display(pd.DataFrame(rows).set_index("strategy").round(3))
print("👉 『全部猜沒病』的準確率就有 85％，recall（有病的人抓到幾成）卻是 0。罕見疾病不能只看 accuracy。")
print("   class weight、過採樣可以多抓到一些病人（recall 上升），代價通常是 precision 下降（誤報變多）。要選哪個，取決於漏診與誤報哪個代價更大。")

In [ ]:
#@title 3-8　總結：每一種前處理，到底值多少？
res = pd.DataFrame(RESULTS)
display(res.round(3))
plt.figure(figsize=(7, 3)); plt.barh(res["label"][::-1], res["acc"][::-1], xerr=res["acc_sd"][::-1], color="#3182bd")
plt.xlim(0.5, 1); plt.xlabel("cross-validated accuracy (mean ± sd)"); plt.tight_layout(); plt.show()
print("標準化的影響最大；加入類別欄位、one-hot 次之；擴增在這裡沒有顯著差異。")
print("前處理沒有『一定有用』的招式，每一招都要用乾淨的驗證方式檢查。")

---
## 單元 4｜推論：新病人進來了

**訓練（training）** 是在學習；**推論（inference）** 是真的拿來用。
關鍵原則：**推論時對資料做的前處理，必須跟訓練時一模一樣。**
這表示：所有「從訓練資料算出來的東西」（標準化的平均與標準差、one-hot 的欄位順序、補缺值用的中位數）都要**一起存起來**，跟模型成套使用。

In [ ]:
#@title 4-1　定案：用訓練＋驗證資料建立前處理、訓練最終模型，一起存起來
import joblib
fill = {**{c: trainval_df[c].median() for c in ALL_NUMERIC},                  # 補缺值用的數字：只用訓練＋驗證資料算
        **{c: trainval_df[c].mode()[0] for c in ["sex", "chest_pain", "thal"]}}

final_prep = make_prep("onehot", True)
X_final = final_prep.fit_transform(add_male(trainval_df)).astype("float32")
y_final = (trainval_df["disease"] == "yes").astype("float32").values
final_model, _ = train_mlp(X_final, y_final, seed=0)

ART = os.path.join(WORKDIR, "artifacts"); os.makedirs(ART, exist_ok=True)
joblib.dump({"prep": final_prep, "fill": fill}, os.path.join(ART, "preprocessor.joblib"))
torch.save(final_model.state_dict(), os.path.join(ART, "model.pt"))
print("✅ 已存到 Drive：", sorted(os.listdir(ART)))
print(f"模型輸入維度：{X_final.shape[1]} 個特徵（原始 {len(FEATURES_V1)} 個數字欄位，經過 one-hot 展開後變多了）")

In [ ]:
#@title 4-2　🔓 終於打開測試集（只能這一次！）
test_locked = pd.read_csv(os.path.join(WORKDIR, "heart_test_locked.csv"))
test_locked[ALL_NUMERIC] = test_locked[ALL_NUMERIC].astype(float)
X_test = final_prep.transform(add_male(test_locked)).astype("float32")
y_test = (test_locked["disease"] == "yes").astype("float32").values
res = evaluate(final_model, X_test, y_test)
print(f"測試集（{len(y_test)} 位從沒被看過的病人）：accuracy {res['acc']:.1%}　recall {res['recall']:.1%}　precision {res['precision']:.1%}　AUC {res['auc']:.3f}")
print("\n👉 這才是對『新病人』的誠實成績。如果現在回頭調整模型再重測，測試集就變成另一份驗證集，失去意義。")

In [ ]:
#@title 4-3　一位新病人走進診間：原始表單 → 同樣的前處理 → 預測
new_patient = {            # 就像從不同系統匯出來的「原始表單」，有各種不整齊的寫法
    "patient_id": "NEW001", "age": "55", "sex": "Male", "chest_pain": "non-anginal",
    "resting_bp": "135 mmHg", "chol": "6.4", "fbs": "0", "rest_ecg": "0", "max_hr": "140",
    "exang": "0", "oldpeak": "1.2", "slope": "2", "ca": "1", "thal": "reversable",
}
print("原始輸入：", {k: v for k, v in new_patient.items() if k in ["sex", "chest_pain", "resting_bp", "chol", "thal"]}, "…")

def predict_risk(raw, prep=final_prep, model=final_model, fill=fill, tweak=None):
    d = auto_clean(pd.DataFrame([raw]), fill=fill)       # 同一套清理規則（含：mmol/L→mg/dL、寫法統一、補值用『訓練時』的數字）
    if tweak: tweak(d)
    return float(predict_proba(model, prep.transform(add_male(d)).astype("float32"))[0])

risk = predict_risk(new_patient)
cleaned = auto_clean(pd.DataFrame([new_patient]), fill=fill)
print(f"清理後：sex={cleaned.sex[0]}, resting_bp={cleaned.resting_bp[0]:.0f}, chol={cleaned.chol[0]:.0f} mg/dL, thal={cleaned.thal[0]}")
print(f"\n✅ 預測有冠心病的機率：{risk:.1%}")

In [ ]:
#@title 4-4　故意做錯：前處理跟訓練時不一致，會發生什麼事？
def no_unit_fix(d): d.loc[0, "chol"] = float(new_patient["chol"])   # 忘了把 mmol/L 換成 mg/dL（直接用 6.4）

# 錯誤 A：跳過標準化（直接把原始數字餵進去）
X_ok = final_prep.transform(add_male(cleaned)).astype("float32")
raw_vals = cleaned[PREP_NUM].values.astype("float32")
X_noscale = X_ok.copy(); X_noscale[:, :len(PREP_NUM)] = raw_vals
risk_noscale = float(predict_proba(final_model, X_noscale)[0])

# 錯誤 B：忘了單位換算
risk_unit = predict_risk(new_patient, tweak=no_unit_fix)

# 錯誤 C：在新資料上「重新」算平均與標準差（只有一個人，標準差是 0）
sc_wrong = StandardScaler().fit(cleaned[PREP_NUM])
X_resc = X_ok.copy(); X_resc[:, :len(PREP_NUM)] = sc_wrong.transform(cleaned[PREP_NUM])
risk_resc = float(predict_proba(final_model, X_resc)[0])

table = pd.DataFrame({"做法": ["✅ 正確：套用訓練時存下來的前處理", "❌ A. 跳過標準化", "❌ B. 忘了膽固醇單位換算", "❌ C. 在新資料上重新標準化"],
                      "預測機率": [risk, risk_noscale, risk_unit, risk_resc]})
display(table.style.format({"預測機率": "{:.1%}"}).hide(axis="index"))

# 錯誤 D：現場用 get_dummies 做 one-hot（新病人只有一位，欄位數、順序都跟訓練時不一樣）
try:
    X_bad = pd.get_dummies(cleaned[["chest_pain", "thal"]]).astype("float32").values
    print(f"\n❌ D. 現場 get_dummies：只得到 {X_bad.shape[1]} 欄；模型需要 {X_final.shape[1]} 欄。")
    predict_proba(final_model, X_bad)
except Exception as e:
    print("   模型直接報錯：", str(e)[:90], "…")
print("\n👉 最危險的是 A、B、C：程式不會報錯，只是預測默默變得不可靠；A 甚至會讓『高風險病人』看起來像『低風險』。")

### 收尾：把今天帶走

**資料不只是數字，它還有來源與隱私**
- **去識別化**：病歷文字裡的姓名、日期、病歷號；DICOM 影像標頭裡的個資；腦部 MRI 重建後有可能認出人臉。研究前要經 IRB，並依個資法處理。
- **偏誤**：資料若只來自某一家醫院、某一族群，模型在別的地方可能就不準。跨院、跨儀器的外部驗證很重要。
- **標籤品質**：不同醫師對同一份 EEG 判讀不一致時，AI 學到的是「誰的標準」？

**閱讀 AI 論文時的檢查清單**
1. 資料從哪裡來？有多少醫院、多少病人？
2. 切分是以「病人」還是以「影像／切片」為單位？
3. 前處理（標準化、補值、挑特徵）是在切分**之前**還是**之後**做的？
4. 有沒有外部驗證（不同醫院、不同儀器）？
5. 標籤是誰標的？有幾位？一致性如何？
6. 評估指標適合這個疾病的盛行率嗎？（別只看 accuracy）

🎉 恭喜完成！你的 Google 雲端硬碟資料夾 `AI_preprocessing_class` 裡，有今天產生的所有檔案。